# Лабораторная работа №2
## Коллекции и управление программой: платёжный шлюз

### Итог работы

Вы обработаете пакет платёжных событий: найдёте дубликаты и
некорректные записи, нормализуете данные, примените правила
платёжного шлюза, измените балансы, сформируете очередь ручной
проверки и построите отчёт смены. В финале нужно самостоятельно
реализовать второй сценарий с дополнительным velocity-лимитом.

### Новые инструменты

`list`, `tuple`, `dict`, `set`, индексы и срезы, `for`, `while`,
`if`/`elif`/`else`, `match`, `break`, `continue`, comprehensions,
`enumerate`, методы коллекций и сортировка.

Функции, классы, файлы, NumPy и Pandas пока не используются. Это
намеренное ограничение: цель работы — увидеть состояние алгоритма
и движение данных внутри циклов.

### Правило выполнения

Запускайте ячейки сверху вниз и не изменяйте исходные наборы
`raw_events` и `capstone_events`. Перед сдачей выполните **Restart
Kernel and Run All Cells**. Все `assert` должны пройти.


## 1. Выбор коллекции

| Структура | Что хранит | Когда выбирать |
|---|---|---|
| `list` | упорядоченные элементы, повторы разрешены | очередь и журнал |
| `tuple` | неизменяемую последовательность | границы и составной ключ |
| `dict` | пары ключ → значение | баланс по номеру счёта |
| `set` | уникальные элементы | дубликаты и стоп-листы |

В этой работе одна и та же операция представлена словарём, пакет
операций — списком, стоп-лист — множеством, а границы пакета —
кортежем.


In [ ]:
initial_balances = {
    "A100": 25_000.0,
    "A200": 15_000.0,
    "A300": 7_000.0,
    "A400": 5_000.0,
}
daily_limits = {
    "A100": 12_000.0,
    "A200": 8_000.0,
    "A300": 5_000.0,
    "A400": 3_000.0,
}
blocked_accounts = {"A400"}
blocked_merchants = {"SCAM-SHOP"}

raw_events = [
    {"id": "E001", "kind": "deposit", "account": "A100", "amount": 5000.0,
     "merchant": None, "city": "Moscow", "counterparty": None},
    {"id": "E002", "kind": "purchase", "account": "A100", "amount": 3200.0,
     "merchant": " books ", "city": "moscow", "counterparty": None},
    {"id": "E003", "kind": "purchase", "account": "A200", "amount": 6500.0,
     "merchant": "tech", "city": " kazan ", "counterparty": None},
    {"id": "E004", "kind": "purchase", "account": "A200", "amount": 2500.0,
     "merchant": "CAFE", "city": "Kazan", "counterparty": None},
    {"id": "E005", "kind": "withdrawal", "account": "A300", "amount": 1500.0,
     "merchant": None, "city": "Moscow", "counterparty": None},
    {"id": "E006", "kind": "purchase", "account": "A400", "amount": 1000.0,
     "merchant": "BOOKS", "city": "Moscow", "counterparty": None},
    {"id": "E007", "kind": "refund", "account": "A100", "amount": 700.0,
     "merchant": "BOOKS", "city": "Moscow", "counterparty": None},
    {"id": "E008", "kind": "purchase", "account": "A100", "amount": 450.0,
     "merchant": "SCAM-SHOP", "city": "Moscow", "counterparty": None},
    {"id": "E009", "kind": "transfer", "account": "A300", "amount": 4000.0,
     "merchant": None, "city": "Moscow", "counterparty": "A200"},
    {"id": "E010", "kind": "cashout", "account": "A300", "amount": 500.0,
     "merchant": None, "city": "Moscow", "counterparty": None},
    {"id": "E002", "kind": "purchase", "account": "A300", "amount": 50.0,
     "merchant": "CAFE", "city": "Moscow", "counterparty": None},
    {"id": "E012", "kind": "purchase", "account": "A100", "amount": -200.0,
     "merchant": "BOOKS", "city": "Moscow", "counterparty": None},
    {"id": "E013", "kind": "purchase", "account": "A999", "amount": 100.0,
     "merchant": "BOOKS", "city": "Moscow", "counterparty": None},
]

# В записях только скалярные значения, поэтому для контроля
# неизменности достаточно скопировать каждый внутренний словарь.
raw_snapshot = [event.copy() for event in raw_events]
print(f"Получено событий: {len(raw_events)}")


### Задание 1. Инвентаризация пакета

Не изменяя `raw_events`, получите:

- `event_ids` — список ID в исходном порядке;
- `unique_event_ids` — множество уникальных ID;
- `duplicate_ids` — множество повторяющихся ID;
- `accounts_seen` — множество всех встреченных счетов;
- `batch_bounds` — кортеж из первого и последнего ID;
- `first_three_ids` — первые три ID с помощью среза.

Здесь намеренно используйте обычные циклы: важно увидеть, как
постепенно заполняется изменяемая коллекция.


In [ ]:
event_ids = []
unique_event_ids = set()
duplicate_ids = set()
accounts_seen = set()
batch_bounds = None
first_three_ids = []
# YOUR CODE HERE


## 2. Контроль качества данных

Внешний пакет нельзя сразу применять к балансам. Сначала нужно
отделить структурные ошибки от бизнес-решений. Оператор `continue`
завершает текущую итерацию и переходит к следующей записи — это
удобно для отбраковки.

Не удаляйте элементы из списка во время `for`: индексы сдвигаются,
и часть записей легко пропустить. Вместо этого формируйте новый
список `clean_events`.


### Задание 2. Валидация и нормализация

Обойдите `raw_events` по порядку. Для каждой записи примените первую
подходящую причину брака:

1. повторный ID → `duplicate_id`;
2. `amount` не является обычным `int`/`float` или не положителен →
   `invalid_amount`;
3. счёт отсутствует в `initial_balances` → `unknown_account`;
4. у перевода неизвестный получатель → `unknown_counterparty`.

Некорректные записи добавляйте в `invalid_events` в формате
`{"id": ..., "reason": ...}`. Корректную запись сначала копируйте,
затем приведите `kind`, `merchant` и `city` к верхнему регистру без
внешних пробелов. `None` у merchant должен остаться `None`.

Заполните `seen_ids`, `clean_events` и `invalid_events`. Исходный
пакет должен остаться неизменным.


In [ ]:
seen_ids = set()
clean_events = []
invalid_events = []
# YOUR CODE HERE


## 3. Comprehensions и производные коллекции

Comprehension подходит для короткого преобразования или отбора без
сложного состояния. Если внутри требуется несколько веток,
изменение баланса или объяснение причины решения, обычный цикл
читается лучше.

Общая форма: `[выражение for элемент in источник if условие]`.
Аналогично создаются множества и словари.


### Задание 3. Операционный обзор

По `clean_events` создайте:

- `active_accounts` — отсортированный список незаблокированных
  счетов с положительным балансом;
- `merchant_catalog` — множество непустых merchant;
- `high_value_ids` — ID событий на сумму не меньше 5000 в исходном
  порядке;
- `kind_counts` — словарь количества событий каждого вида.

Первые три результата получите comprehensions. Для `kind_counts`
сначала создайте словарь с нулями, затем накопите значения циклом.


In [ ]:
active_accounts = []
merchant_catalog = set()
high_value_ids = []
kind_counts = {}
# YOUR CODE HERE


## 4. Состояние и порядок правил

Теперь каждая операция зависит от результатов предыдущих. Это
**состояние-зависимый алгоритм**: перестановка событий может изменить
итоговые балансы и решения.

Порядок проверок является частью спецификации:

1. заблокированный счёт;
2. запрещённый merchant;
3. поддерживаемый вид операции;
4. дневной лимит для списаний;
5. достаточный баланс;
6. применение операции.

`match` определяет смысл вида операции, а `if` проверяет правила.
Сумма перевода списывается у отправителя и зачисляется получателю.
Операции со статусом `REVIEW` и `REJECTED` пока не меняют баланс.


### Задание 4. Процессор платёжных событий

Создайте независимые копии балансов и нулевой расход по каждому
счёту. Обработайте все `clean_events`.

Для каждого события добавьте в `decisions` словарь с ключами:
`id`, `account`, `kind`, `amount`, `status`, `reason`,
`balance_after`.

Возможные статусы: `APPROVED`, `REVIEW`, `REJECTED`. Причины:
`ok`, `blocked_account`, `blocked_merchant`, `unsupported_kind`,
`daily_limit`, `insufficient_funds`.

События `REVIEW` (если превышен дневной лимит) копируйте в `review_queue`. Дневной расход
увеличивают только одобренные `purchase`, `withdrawal`, `transfer`.
`deposit` и `refund` увеличивают баланс и не расходуют лимит.


In [ ]:
final_balances = initial_balances.copy()
daily_spent = {account: 0.0 for account in initial_balances}
decisions = []
review_queue = []
# YOUR CODE HERE


## 5. Вложенная агрегация

Журнал решений удобен для аудита, но руководителю нужны итоги.
Словарь может содержать другие словари: например,
`stats_by_status[status]["amount"]`. Метод `setdefault` создаёт
начальное значение только при отсутствии ключа.


### Задание 5. Статистика решений


За один проход по `decisions` постройте:

- `stats_by_status`: для каждого статуса количество `count` и сумма
  `amount`;
- `reason_counts`: количество решений по каждой причине;
- `review_accounts`: множество счетов со статусом `REVIEW`;
- `rejected_accounts`: множество счетов со статусом `REJECTED`.

Не задавайте имена статусов заранее: структура должна заполняться
по данным.


In [ ]:
stats_by_status = {}
reason_counts = {}
review_accounts = set()
rejected_accounts = set()
# YOUR CODE HERE


## 6. Очередь и цикл `while`

`for` удобен, когда известна коллекция для обхода. `while` подходит,
когда завершение определяется состоянием: очередь пуста, исчерпан
лимит или достигнута контрольная точка.

Для учебного списка используем `pop(0)`. У больших очередей такая
операция дорогая; с `collections.deque` познакомимся в работе со
стандартной библиотекой.


### Задание 6. Ручная проверка

Оператор может вручную одобрить превышения суммарно не более чем на
3000 RUB. Создайте копии `final_balances`, `daily_spent` и
`review_queue`.

Пока очередь не пуста:

- извлеките первый элемент;
- если его сумма превышает остаток ручного лимита, поместите ID
  текущего и всех оставшихся элементов в `deferred_review_ids` и
  завершите цикл через `break`;
- иначе примените списание или перевод, обновите дневной расход и
  лимит, добавьте ID в `manual_approved_ids`.

Исходная `review_queue` измениться не должна.


In [ ]:
settled_balances = final_balances.copy()
settled_spent = daily_spent.copy()
pending_review = [event.copy() for event in review_queue]
manual_budget = 3_000.0
manual_approved_ids = []
deferred_review_ids = []
# YOUR CODE HERE


## 7. Сортировка и отчёт

`sorted` умеет сравнивать кортежи: сначала по первому элементу,
затем по второму. Это позволяет построить рейтинг без отдельной
функции-ключа. `enumerate(..., start=1)` добавляет место в рейтинге.


### Задание 7. Отчёт смены

Сформируйте `spending_ranking` — список кортежей `(account, spent)`
по убыванию расхода. Затем создайте строки с номером места и
объедините их переводами строк в `shift_report`.

Первая строка должна быть `Рейтинг дневного расхода:`. Суммы
форматируйте с двумя знаками после запятой и разделителем тысяч.


In [ ]:
spending_ranking = []
ranking_lines = []
shift_report = ""
# YOUR CODE HERE


## 8. Итоговый мини-проект

Вторая смена — независимый набор данных. Здесь появляется
**velocity-лимит**: число одобренных списаний со счёта не должно
превышать заданный максимум. Его проверяют после дневного лимита,
но до достаточности баланса.

### Задание 8. Ночная смена

1. Убедитесь, что ID уникальны (`capstone_ids_unique`).
2. Создайте копию балансов, нулевые расходы и счётчики списаний.
3. Обработайте события по правилам задания 4, добавив velocity-
   проверку. Заблокированный merchant проверяется первым.
4. Перевод меняет два баланса, но расход и число списаний относятся
   только к отправителю.
5. Постройте `capstone_status_counts`, списки review/rejected ID,
   множество риск-счетов и `capstone_health` (`LOW`, если баланс
   меньше 3000 RUB, иначе `STABLE`).
6. Сформируйте итоговую строку `capstone_report` заданного формата.

В `capstone_decisions` достаточно ключей `id`, `account`, `status`,
`reason`, `amount`. Операции `REVIEW`/`REJECTED` баланс не меняют.


In [ ]:
capstone_initial_balances = {"C100": 9_000.0, "C200": 4_000.0}
capstone_daily_limits = {"C100": 5_000.0, "C200": 2_500.0}
capstone_velocity_limits = {"C100": 2, "C200": 2}
capstone_blocked_merchants = {"CASINO"}
capstone_events = [
    {"id": "C001", "kind": "purchase", "account": "C100",
     "amount": 1200.0, "merchant": "BOOKS", "counterparty": None},
    {"id": "C002", "kind": "withdrawal", "account": "C200",
     "amount": 500.0, "merchant": None, "counterparty": None},
    {"id": "C003", "kind": "purchase", "account": "C100",
     "amount": 4100.0, "merchant": "TECH", "counterparty": None},
    {"id": "C004", "kind": "refund", "account": "C200",
     "amount": 200.0, "merchant": "BOOKS", "counterparty": None},
    {"id": "C005", "kind": "purchase", "account": "C200",
     "amount": 300.0, "merchant": "CASINO", "counterparty": None},
    {"id": "C006", "kind": "transfer", "account": "C200",
     "amount": 1800.0, "merchant": None, "counterparty": "C100"},
    {"id": "C007", "kind": "purchase", "account": "C200",
     "amount": 100.0, "merchant": "CAFE", "counterparty": None},
    {"id": "C008", "kind": "deposit", "account": "C100",
     "amount": 1000.0, "merchant": None, "counterparty": None},
]


In [ ]:
capstone_ids_unique = None
capstone_balances = capstone_initial_balances.copy()
capstone_spent = {account: 0.0 for account in capstone_balances}
capstone_debit_counts = {account: 0 for account in capstone_balances}
capstone_decisions = []

capstone_status_counts = {}
capstone_review_ids = []
capstone_rejected_ids = []
capstone_risk_accounts = set()
capstone_health = {}
capstone_report = ""
# YOUR CODE HERE


## Вывод и защита

Добавьте после этой ячейки собственный вывод на 7–10 предложений:

1. Почему порядок событий влияет на результат?
2. Почему E004 отправлена на REVIEW, хотя денег на счёте достаточно?
3. Почему E009 не изменила два баланса автоматически?
4. Чем ошибка входных данных отличается от `REJECTED`?
5. Почему для `review_queue` понадобилась копия?
6. Как сработал velocity-лимит во второй смене?
7. Назовите хотя бы один инвариант системы.

### Контрольные вопросы

1. Когда нужен список, а когда множество?
2. Почему ключ словаря должен быть хешируемым?
3. Чем `append` отличается от `extend`?
4. Почему опасно удалять элементы списка внутри `for` по нему?
5. Когда `continue` делает код понятнее?
6. Чем `while` отличается от `for` в задаче с очередью?
7. Как `match` и `if` разделяют разные виды решений?
8. Что означает «не изменять исходные данные»?
9. Почему проверка лимита должна происходить до изменения баланса?
10. Когда comprehension хуже обычного цикла?


_Напишите здесь собственный вывод._
